
# OpenWebUI Multi‑Agent Goal‑Oriented Workflow (Colab Notebook)

This notebook demonstrates a **modular, inspectable, and reproducible** multi‑agent system that coordinates several agent roles (Planner, Researcher, Coder, Critic, and Synthesizer) to work toward a user‑defined goal.  
It uses the **OpenWebUI API** (OpenAI‑compatible) via the endpoints:
- `POST {API_BASE}/chat/completions` for chat generation
- `POST {API_BASE}/files` (optional) for file uploads

**Key features**  
- Clear separation of concerns (each role has a targeted system prompt)  
- Blackboard memory for artifacts, decisions, and intermediate outputs  
- Deterministic step loop with configurable limits, temperature, and stop criteria  
- Minimal dependency footprint (pure `requests`)  
- Optional file upload integration for server‑side retrieval or inline context

> Tip: This notebook mirrors the request/response schema from your existing OpenWebUI client and extends it into a robust, multi‑agent workflow.



## 1. Environment check and lightweight helpers

We verify the environment and define minimal utilities (printing, JSON pretty‑print, and HTTP helpers with optional bearer token).


In [38]:

import sys, platform, subprocess, json, time, os, typing, dataclasses, re
from typing import List, Dict, Any, Optional
import requests
from getpass import getpass
import traceback
from dataclasses import dataclass, field

def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version)
print("Platform:", platform.platform())

def pretty_print_json(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False)[:20000])

def headers_with_auth(api_key: Optional[str] = None, json_mode: bool = True) -> Dict[str, str]:
    h = {"Content-Type": "application/json"} if json_mode else {}
    if api_key:
        h["Authorization"] = f"Bearer {api_key}"
    return h

def post_json(url: str, payload: Dict[str, Any], api_key: Optional[str] = None, timeout: int = 120) -> requests.Response:
    return requests.post(url, headers=headers_with_auth(api_key, json_mode=True), json=payload, timeout=timeout)

def post_multipart(url: str, files: Dict[str, Any], data: Optional[Dict[str, str]] = None, api_key: Optional[str] = None, timeout: int = 120) -> requests.Response:
    return requests.post(url, headers=headers_with_auth(api_key, json_mode=False), files=files, data=data or {}, timeout=timeout)


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35



## 2. Configuration

Set your API base, key, and default model. The API base should be the **OpenWebUI v1 base**, e.g. `https://<host>/api/v1`.


In [18]:

# === REQUIRED: Set your server info ===
API_BASE    = "https://ursinus.ai/api/v1"   # Example; replace with your server base
API_KEY     = getpass("Enter your OpenWebUI API Key (or leave empty if not required): ")
MODEL       = "gpt-3.5-turbo"               # Replace with your deployed model
TIMEOUT     = 120
TEMP        = 0.2
EXEC_DRYRUN = False
MAX_PROMPTS = 100

assert isinstance(API_BASE, str) and API_BASE.strip(), "Please set API_BASE"
assert isinstance(MODEL, str) and MODEL.strip(), "Please set MODEL"
print("Configured. API_BASE=", API_BASE, "| MODEL=", MODEL)


Enter your OpenWebUI API Key (or leave empty if not required): ··········
Configured. API_BASE= https://ursinus.ai/api/v1 | MODEL= gpt-3.5-turbo



## 3. Optional: Upload a file to the server

If your server exposes a compatible **`/files`** endpoint, you can upload a file and pass its file ID into downstream prompts.  
If not, you can inline a truncated document text as user content instead.


In [19]:

# Supply a local path (in Colab, you can upload via the Files pane or python I/O)
LOCAL_FILEPATH = ""  # e.g., "/content/sample.pdf"
uploaded_file_id = None
files_api_supported = False

if LOCAL_FILEPATH and os.path.exists(LOCAL_FILEPATH):
    try:
        url_files = API_BASE.rstrip("/") + "/files"
        with open(LOCAL_FILEPATH, "rb") as fh:
            resp = post_multipart(
                url_files,
                files={"file": (os.path.basename(LOCAL_FILEPATH), fh)},
                data={"purpose": "assistants"},
                api_key=API_KEY,
                timeout=TIMEOUT
            )
        if resp.ok:
            files_api_supported = True
            data = resp.json()
            uploaded_file_id = data.get("id")
            print("Files API success. file_id:", uploaded_file_id)
        else:
            print("Files API not available or failed:", resp.status_code, resp.text[:400])
    except requests.exceptions.RequestException as e:
        print("Files API request exception:", e)
else:
    print("No file provided; skipping /v1/files.")


No file provided; skipping /v1/files.



## 4. Minimal OpenWebUI chat client

We wrap the `POST /chat/completions` call with sensible defaults, optional retry, and a uniform response schema.


In [44]:

num_prompts = 0

def chat(
    messages: List[Dict[str, str]],
    model: str = MODEL,
    temperature: float = TEMP,
    api_base: str = API_BASE,
    api_key: Optional[str] = API_KEY,
    timeout: int = TIMEOUT,
    max_retries: int = 2,
) -> Dict[str, Any]:
    global num_prompts

    url = api_base.rstrip("/") + "/chat/completions"
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "stream": False
    }
    last_exc = None
    for attempt in range(max_retries + 1):
        try:
            if num_prompts < MAX_PROMPTS:
              num_prompts += 1
              print(f"Prompting with: {payload}")
              resp = post_json(url, payload, api_key=api_key, timeout=timeout)
              if resp.ok:
                  data = resp.json()
                  # OpenAI-style extraction
                  content = None
                  try:
                      content = data["choices"][0]["message"]["content"]
                  except Exception:
                      pass
                  print(f"Result: {data}")
                  return {"ok": True, "content": content, "raw": data, "request": payload}
              else:
                  last_exc = RuntimeError(f"HTTP {resp.status_code}: {resp.text[:400]}")
            else:
              print("Max prompts exceeded...")
              return {"ok": False, "error": "Max prompts exceeded", "request": payload}
        except requests.exceptions.RequestException as e:
            last_exc = e
        time.sleep(0.8 * (attempt + 1))
    return {"ok": False, "error": str(last_exc), "request": payload}



## 5. Blackboard memory

A simple blackboard holds the **goal**, **tasks**, **artifacts**, and **decisions** any agent can read/write.


In [39]:

@dataclass
class Blackboard:
    goal: str
    tasks: list = field(default_factory=list)       # queue of next actions
    artifacts: dict = field(default_factory=dict)   # named outputs
    decisions: list = field(default_factory=list)   # audit trail
    notes: list = field(default_factory=list)       # scratch

    def snapshot(self) -> str:
        # Compact textual view fed into agent prompts
        try:
            # Serialize artifacts safely: content too, not just keys
            artifacts_summary = {
                k: (v if isinstance(v, str) else json.dumps(v, ensure_ascii=False, indent=2))
                for k, v in self.artifacts.items()
            }

            return json.dumps({
                "goal": self.goal,
                "tasks": self.tasks,
                "artifacts": artifacts_summary,
                "last_decision": self.decisions[-1] if self.decisions else None,
                "notes_tail": self.notes[-5:]
            }, ensure_ascii=False)
        except Exception as e:
            print(f"[Blackboard.snapshot] {e}")
            traceback.print_exc()
            # fallback to something minimal if serialization fails
            return json.dumps({
                "goal": self.goal,
                "tasks": self.tasks,
                "artifacts": list(self.artifacts.keys()),
                "last_decision": self.decisions[-1] if self.decisions else None,
                "notes_tail": self.notes[-5:]
            }, ensure_ascii=False)



## 6. Agent roles

Each agent uses a focused system prompt and receives the **blackboard snapshot** plus recent outputs.  
Agents return **structured directives** that the orchestrator interprets.


In [42]:

ROLE_PROMPTS = {
    "Planner": (
        "You are the Planner. Given a goal and current state, decompose work into clear, atomic tasks."
        "Return a prioritized JSON list under key 'next_tasks'. Include 1-3 tasks only. Avoid redundancy."
    ),
    "Researcher": (
        "You are the Researcher. For the assigned task, produce concise findings.\n"
        "IMPORTANT SCHEMA FACTS (authoritative):\n"
        "- Endpoint: POST {API_BASE}/chat/completions (OpenAI-compatible)\n"
        "- Required JSON keys: model (string), messages (array of {role, content})\n"
        "- Optional keys: temperature (float), stream (bool)\n"
        "- There is NO single 'prompt' parameter; use 'messages' instead.\n"
        "Return JSON with keys: 'findings' (markdown), and 'confidence' (0-1)."
    ),
    "Coder": (
        "You are the Coder. For the assigned task, produce robust, minimal Python code and a short explanation.\n"
        "Use the authoritative schema:\n"
        "  url = API_BASE.rstrip('/') + '/chat/completions'\n"
        "  payload = { 'model': MODEL, 'messages': [{'role':'system','content':'...'},"
        "                                           {'role':'user','content':'...'}],"
        "              'temperature': TEMP, 'stream': False }\n"
        "  headers must include Authorization: Bearer <API_KEY> when required and Content-Type: application/json\n"
        "Return JSON with keys: 'code' (fenced ```python block) and 'notes' (bullets)."
    ),
    "Critic": (
        "You are the Critic. Inspect RESEARCH_JSON and CODE_JSON for correctness and alignment with the goal.\n"
        "Return valid JSON with keys:\n"
        "  'issues': array of short bullet strings,\n"
        "  'severity': 'low'|'medium'|'high',\n"
        "  'go_no_go': 'go'|'revise',\n"
        "  'corrective_task': a single actionable task string if go_no_go='revise', else ''."
    ),
    "Synthesizer": (
        "You are the Synthesizer. "
        "Produce a self-contained, publication-ready deliverable that integrates all prior work (tasks, decisions, artifacts, notes), "
        "not a mere recap.\n\n"

        "Non-negotiable requirements:\n"
        "- Use and cite prior work by artifact key or decision index (e.g., [artifact: code_round3], [decision: 12]).\n"
        "- Prefer concrete content (code, numbered steps, parameter values) over generalities.\n"
        "- If data is missing, write MISSING and explain how to obtain it.\n"
        "- Do not invent APIs, endpoints, or data not present in the input unless explicitly allowed by the goal.\n\n"

        "Structure and depth:\n"
        "- Write in Markdown with these exact top-level headings:\n"
        "  1. Objective\n"
        "  2. Inputs Reviewed\n"
        "  3. Integrated Deliverable\n"
        "  4. Implementation Notes & Rationale\n"
        "  5. Validation & Next Steps\n"
        "  6. Appendix: Source Excerpts\n\n"

        "Content policy:\n"
        "- 'Integrated Deliverable' must be the final artifact requested by the goal (e.g., a tutorial with runnable code). "
        "It must be self-contained (copy–paste runnable code, explicit parameters, curl examples if relevant).\n"
        "- 'Inputs Reviewed' must list all artifacts and decisions actually used, with 1–2 bullet summaries and bracket citations.\n"
        "- 'Implementation Notes & Rationale' must trace choices back to inputs (e.g., why temperature=0.2).\n"
        "- 'Validation & Next Steps' must include how to test/run, expected outputs, and next improvements.\n"
        "- 'Appendix: Source Excerpts' must include short verbatim snippets (≤80 words) from artifacts or notes supporting key claims.\n\n"

        "Style:\n"
        "- Concise but complete; no filler. No meta-commentary about being an AI.\n\n"

        "If the goal involves code:\n"
        "- Use fenced ```python blocks with imports and minimal .env handling.\n"
        "- Include request/response parsing and error handling for non-200 responses.\n"
        "- Provide one minimal end-to-end example first, then optional variants.\n\n"

        "Output:\n"
        "- Return only the Markdown document wrapped in json."
    )
}

def as_messages(system_prompt: str, user_prompt: str) -> list:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

def llm_json(role: str, user_prompt: str) -> Dict[str, Any]:
    msgs = as_messages(ROLE_PROMPTS[role], user_prompt)
    out = chat(msgs)
    if not out.get("ok"):
        return {"_error": out.get("error")}
    # Try to find a JSON object in the model output
    txt = out["content"] or ""
    m = re.search(r"\{[\s\S]*\}", txt)
    if not m:
        return {"_raw": txt.strip()}
    try:
        return json.loads(m.group(0))
    except Exception:
        return {"_raw": txt.strip()}



## 7. Orchestrator

A simple deterministic loop:
1. Planner proposes up to 3 tasks.
2. For each task, run Researcher → Coder (optional, when task looks like coding) → Critic.
3. If Critic says **go**, persist artifacts and proceed. If **revise**, add a corrective task.
4. After N steps or if the goal appears satisfied, the Synthesizer produces a final brief.


In [33]:
def execute_step(
    task: str,
    bb: "Blackboard",
    *,
    dryrun: bool = True,
    role: str = "Executor",
    api_base: str = API_BASE,
    api_key: str | None = API_KEY,
    timeout: int = TIMEOUT,
) -> dict:
    """
    Execute a task by prompting the model with the task name and
    the most relevant prior results from the blackboard.

    If dryrun=True, returns immediately with a no-op result.

    Returns dict with keys:
      - ok (bool)
      - content (model response or stub)
      - role (executor role used)
      - used_artifacts (artifact keys fed into prompt)
    """
    import json, traceback, time

    if dryrun:
        print(f"Would execute {task}")

        return {
            "ok": True,
            "dryrun": True,
            "role": role,
            "note": f"Dry run: would have executed task '{task}'",
            "timestamp": time.time(),
        }

    try:
        print(f"Executing {task}")

        # --- Gather the most relevant prior artifacts
        used_artifacts = {
            k: v for k, v in bb.artifacts.items()
            if any(x in k for x in ["research", "code", "critic", "exec"])
        }
        context_text = json.dumps(used_artifacts, ensure_ascii=False)[:6000]

        # --- Construct a single execution prompt
        system_prompt = (
            "You are the Executor agent. Given a task name, the project goal, "
            "and contextual prior results, carry out the task as precisely as possible. "
            "Return valid JSON with keys 'result' (summary or data) and 'notes'."
        )

        user_prompt = (
            f"GOAL: {bb.goal}\n"
            f"TASK: {task}\n"
            f"STATE: {bb.snapshot()}\n"
            f"PRIOR RESULTS:\n{context_text}\n"
            f"Return JSON only."
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        url = api_base.rstrip("/") + "/chat/completions"
        payload = {
            "model": MODEL,
            "messages": messages,
            "temperature": TEMP,
            "stream": False,
        }

        resp = post_json(url, payload, api_key=api_key, timeout=timeout)
        if not resp.ok:
            return {"ok": False, "status": resp.status_code, "text": resp.text[:400]}

        data = resp.json()
        content = None
        try:
            content = data["choices"][0]["message"]["content"]
        except Exception:
            pass
        return {
            "ok": True,
            "role": role,
            "content": content,
            "used_artifacts": list(used_artifacts.keys()),
        }

    except Exception as e:
        traceback.print_exc()
        return {"ok": False, "error": f"execute_step: {e}"}

In [40]:
def is_codey(task) -> bool:
    """
    Returns True if the task looks like a coding or implementation step.
    Accepts either strings or structured objects from the planner.
    """
    # Convert dicts or other objects to text for inspection
    if isinstance(task, dict):
        task_text = json.dumps(task, ensure_ascii=False)
    else:
        task_text = str(task)
    return any(k in task_text.lower() for k in ["code", "script", "notebook", "function", "api client", "implement"])

def _normalize_tasks(proposed):
    norm = []
    for t in (proposed or []):
        if isinstance(t, dict) and "task" in t:
            norm.append(str(t["task"]))
        else:
            norm.append(str(t))
    return [s.strip() for s in norm if s and s.strip()]

def orchestrate(goal: str, max_rounds: int = 4) -> Blackboard:
    bb = Blackboard(goal=goal)
    bb.decisions.append({"event": "init", "goal": goal, "ts": time.time()})

    for round_idx in range(1, max_rounds + 1):
        # 1) Plan
        plan_prompt = (
            f"GOAL: {bb.goal}\n"
            f"STATE: {bb.snapshot()}\n"
            f"Return only JSON."
        )

        plan = llm_json("Planner", plan_prompt)
        proposed = _normalize_tasks(plan.get("next_tasks", []))
        bb.decisions.append({"event": "plan", "round": round_idx, "tasks": proposed})
        if not proposed:
            bb.notes.append("Planner returned no tasks; stopping.")
            break

        if not proposed:
            bb.notes.append("Planner returned no tasks; stopping.")
            break

        for task in proposed:
            # 2) Research
            research_prompt = (
                f"GOAL: {bb.goal}\nTASK: {task}\nSTATE: {bb.snapshot()}\n"
                f"If a file id exists, you may assume the server can retrieve it: file_id={{{uploaded_file_id}}}.\n"
                f"Return only JSON."
            )
            R = llm_json("Researcher", research_prompt)
            bb.artifacts[f"research_round{round_idx}"] = R
            bb.decisions.append({"event": "research", "task": task, "result_keys": list(R.keys())})

            # 3) Execute task based on prior outputs
            E = execute_step(task, bb, dryrun=EXEC_DRYRUN)
            bb.artifacts[f"exec_round{round_idx}"] = E
            bb.decisions.append({"event": "execute", "task": task, "ok": E.get("ok")})

            # 4) Code (when relevant)
            if is_codey(task):
                code_prompt = (
                    f"GOAL: {bb.goal}\nTASK: {task}\nSTATE: {bb.snapshot()}\n"
                    f"Generate minimal, runnable code with a short explanation. Return only JSON."
                )
                C = llm_json("Coder", code_prompt)
                bb.artifacts[f"code_round{round_idx}"] = C
                bb.decisions.append({"event": "code", "task": task, "result_keys": list(C.keys())})

            # 5) Critic
            latest_R = bb.artifacts.get(f"research_round{round_idx}", {})
            latest_C = bb.artifacts.get(f"code_round{round_idx}", {})
            critic_prompt = (
                f"GOAL: {bb.goal}\n"
                f"TASK: {task}\n"
                f"RESEARCH_JSON: {json.dumps(latest_R, ensure_ascii=False)}\n"
                f"CODE_JSON: {json.dumps(latest_C, ensure_ascii=False)}\n"
                f"Return only JSON."
            )

            K = llm_json("Critic", critic_prompt)
            bb.artifacts[f"critic_round{round_idx}"] = K
            bb.decisions.append({"event": "critic", "task": task, "decision": K.get("go_no_go")})

            if K.get("go_no_go") != "go":
                corrective = K.get("corrective_task") or "Revise previous work to address key issues."
                bb.tasks.append(corrective)

        # brief check for completion signal in artifacts
        if any("final" in k for k in bb.artifacts.keys()):
            break

    # 5) Synthesize
    synth_prompt = f"GOAL: {bb.goal}\nSTATE: {bb.snapshot()}\nReturn only JSON."
    S = llm_json("Synthesizer", synth_prompt)
    bb.artifacts["final_synthesis"] = S
    bb.decisions.append({"event": "synthesize", "ts": time.time()})

    return bb



## 8. Run the multi‑agent loop

Provide a concrete goal. You can run multiple times with different temperatures or models.


In [45]:

USER_GOAL = "Produce a concise tutorial and sample code for using the OpenWebUI /chat/completions API from Python."

bb = orchestrate(USER_GOAL, max_rounds=3)
print("Decisions:")
pretty_print_json(bb.decisions)
print("\nArtifacts summary keys:", list(bb.artifacts.keys()))


Prompting with: {'model': 'gpt-3.5-turbo', 'messages': [{'role': 'system', 'content': "You are the Planner. Given a goal and current state, decompose work into clear, atomic tasks.Return a prioritized JSON list under key 'next_tasks'. Include 1-3 tasks only. Avoid redundancy."}, {'role': 'user', 'content': 'GOAL: Produce a concise tutorial and sample code for using the OpenWebUI /chat/completions API from Python.\nSTATE: {"goal": "Produce a concise tutorial and sample code for using the OpenWebUI /chat/completions API from Python.", "tasks": [], "artifacts": {}, "last_decision": {"event": "init", "goal": "Produce a concise tutorial and sample code for using the OpenWebUI /chat/completions API from Python.", "ts": 1762383030.0265064}, "notes_tail": []}\nReturn only JSON.'}], 'temperature': 0.2, 'stream': False}
Result: {'id': 'chatcmpl-CYgchba1O00NDGuOHoxZqCgFwxcYe', 'object': 'chat.completion', 'created': 1762383031, 'model': 'gpt-3.5-turbo-0125', 'choices': [{'index': 0, 'message': {'


## 9. Inspect and export results

You can inspect the final synthesis and persist the blackboard to JSON for later use.


In [ ]:

final = bb.artifacts.get("final_synthesis", {})
print("Final Synthesis (raw):")
pretty_print_json(final)

# Save the entire run (blackboard) to a JSON file
out_path = "/mnt/data/openwebui_multiagent_run.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump({
        "goal": bb.goal,
        "decisions": bb.decisions,
        "artifacts": bb.artifacts,
        "notes": bb.notes
    }, f, ensure_ascii=False, indent=2)
print("Saved:", out_path)



## 10. Advanced: Customizing roles, routing, and tools

- Modify `ROLE_PROMPTS` to add/remix agents (e.g., **Evaluator**, **Decomposer**, **Executor**).
- Replace `is_codey()` with a classifier prompt to route tasks more intelligently.
- Add lightweight tools (e.g., a Python REPL or HTTP fetch) gated behind explicit prompts.
- Persist the blackboard to a vector store if you want retrieval‑augmented iterations.
